# Retrieve firm data from Zefix

**Due Date: 2025.05.06 23:59**

For this assignment you can work by yourself or in pair.

For the deliverable, submit to icorsi your notebook with the following name: `lastname.ipynb` if you work by yourself or `lastname1_lastname2.ipynb` if you work in pair.

## 1. Find all firms named UBS

The goal of this exercise is to find firms whose name matches "UBS" and for them collect the information exemplified in the table below:

![](firms_table_ubs.png)

The expected output is the information as in the table above, in the format you prefer (Pandas dataframe, csv, json).

To perform a search of firms by name, you can use the following POST endpoint:

```
https://www.zefix.ch/ZefixREST/api/v1/firm/search.json
```

which takes a json with the following forma:

```
{
  "name": "<YOUR SEARCH STRING>",
  "searchType":"exact"
}
```

To get information about a firm once you know the firm ehraid you can use the following GET endpoint

```
https://www.zefix.ch/ZefixREST/api/v1/firm/<EHRAID>
```

which, given a `EHRAID`, provides information about the firm.


Note that in the table above, there are fields named `n_of_SOMETHING` as for example `n_of_branchOffices`. To populate this field, you will need to count the number of branch offices that you will get from the `firm/<EHRAID>` GET endpoint.
There is also a field called `legalForm` (the legal form in English), whereas the firm GET endpoint provides you with a `legalFormId` information. You can retrieve the mapping between ids and legal form names (in various languages), with the following GET endpoint:

```
https://www.zefix.ch/ZefixREST/api/v1/legalForm
```


## 2. Find all firms taken over by Credit Suisse transitively

The goal of the second exercise is to find all the firms that were acquired by Credit Suisse, transitively. This information can be found on Zefix, with the property `hasTakenOver`, which for a given firm X provides a list of all the firms taken over by X.

We want to collect this information transitively, meaning that if a firm X was taken over by Credit Suisse, and another firm Y was taken over by X, in the list we want both X and Y.

```
Credit Suisse <= X <= Y
```

The expected output is a json file structured as follows:

```
[
  {
    "name" : "<NAME OF THE COMPANY>",
    "ehraid": "<EHRA ID OF THE COMPANY>",
    "legalSeat": "<LEGAL SEAT OF THE COMPANY>",
    "legalFormId": "<LEGAL FORM ID OF THE COMPANY>",
    "status": "<STATUS OF THE COMPANY>",
    "cantonalExcerptWeb": "<LINK TO THE CANTONAL EXCERPT>",
    "deleteDate": "<CANCELLATION DATE OF THE COMPANY OR NULL>",
    "cs_hops" : "<NUMBER OF LINKS BETWEEN CREDIT SUISSE AND THE COMPANY>"
  },
  {
  ...
  },
  ...
]
```

The `cs_hops` represents the number of `hasTakenOver` relationships between the company and Credit Suisse. In the example above, the `cs_hops` is 1 for X and 2 for Y.

To perform this exercise you can use the following endpoint:

```
https://www.zefix.ch/ZefixREST/api/v1/firm/<EHRAID>
```

This is a GET endpoint, that given a `EHRAID` provides information about the firm, including the list of companies taken over (field `hasTakenOver`). Note that these are only the direct take over, not the transitive ones.
    
Since there are several Credit Suisse firms, the one to use for the exercise has EHRAID = 388851.
    

In [1]:
import requests
import json
import pandas as pd

# Task1

### 1. Search for UBS Companies
Send a POST request to `https://www.zefix.ch/ZefixREST/api/v1/firm/search.json` with JSON payload.
### 2. Retrieve Legal Form Mapping
Send a GET request to `https://www.zefix.ch/ZefixREST/api/v1/legalForm` and create a mapping dictionary from ID to English name.

### 3. Get Detailed Information for Each UBS Company
Iterate through each company in the search results, send a GET request to `https://www.zefix.ch/ZefixREST/api/v1/firm/<EHRAID>`, and extract necessary fields.

### 4. Summarize and Organize Data
Compute statistics like the number of branches, apply the legal form mapping, and compile all information into a table (DataFrame). Export the final structured data as a pandas DataFrame or any other required format.

In [10]:
# Step 1: Search for all companies whose name contains 'UBS' (with pagination and safe exit)
import requests
import json

search_url = "https://www.zefix.ch/ZefixREST/api/v1/firm/search.json"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

all_companies = []
offset = 0
max_entries = 30  # As observed from the web payload
max_total = 2000  # Safety limit, adjust as needed

while True:
    search_payload = {
        "name": "UBS",
        "searchType": "exact",
        "offset": offset,
        "maxEntries": max_entries
    }
    response = requests.post(search_url, json=search_payload, headers=headers)
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        break
    data = response.json()
    company_list = data.get("list", [])
    all_companies.extend(company_list)
    print(f"Fetched {len(company_list)} companies, total so far: {len(all_companies)}")
    # Exit if no more results or reached safety limit
    if not data.get("hasMoreResults", False):
        print("No more results, exiting loop.")
        break
    if len(all_companies) >= max_total:
        print("Reached safety limit, exiting loop.")
        break
    offset += max_entries

print(f"Total companies found: {len(all_companies)}")

Fetched 28 companies, total so far: 28
No more results, exiting loop.
Total companies found: 28


In [11]:
# Step 2: Retrieve the mapping between legal form IDs and their English names

legalform_url = "https://www.zefix.ch/ZefixREST/api/v1/legalForm"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

response = requests.get(legalform_url, headers=headers)
if response.status_code == 200:
    legal_forms = response.json()
    # Build a mapping from legal form ID to English name
    legal_form_map = {form['id']: form['name']['en'] for form in legal_forms if 'id' in form and 'name' in form and 'en' in form['name']}
    print(f"Retrieved {len(legal_form_map)} legal forms.")
    # Show a sample
    for k in list(legal_form_map.keys())[:3]:
        print(f"ID: {k} -> English name: {legal_form_map[k]}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

Retrieved 19 legal forms.
ID: 1 -> English name: Sole proprietorship
ID: 2 -> English name: General Partnership
ID: 3 -> English name: Corporation


In [12]:
# Step 3: Get detailed information for each UBS company

ubs_detailed_info = []
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

for i, company in enumerate(all_companies):
    ehraid = company.get('ehraid')
    if not ehraid:
        print(f"Skipping company without ehraid: {company.get('name', 'Unknown')}")
        continue
    detail_url = f"https://www.zefix.ch/ZefixREST/api/v1/firm/{ehraid}"
    response = requests.get(detail_url, headers=headers)
    if response.status_code == 200:
        detail_info = response.json()
        ubs_detailed_info.append(detail_info)
        print(f"[{i+1}/{len(all_companies)}] Retrieved: {detail_info.get('name', 'Unknown')}")
    else:
        print(f"[{i+1}/{len(all_companies)}] Failed to retrieve details for ehraid {ehraid}")

print(f"\nRetrieved detailed information for {len(ubs_detailed_info)} UBS companies.")

[1/28] Retrieved: UBS 1e Sammelstiftung
[2/28] Retrieved: UBS AG
[3/28] Retrieved: UBS AG
[4/28] Retrieved: UBS Asset Management AG
[5/28] Retrieved: UBS Asset Management Switzerland AG
[6/28] Retrieved: UBS Business Solutions AG
[7/28] Retrieved: UBS Card Center AG
[8/28] Retrieved: UBS Clean Energy Infrastructure Switzerland 2 AG
[9/28] Retrieved: UBS Clean Energy Infrastructure Switzerland 2 KmGK
[10/28] Retrieved: UBS Europe SE, Frankfurt am Main, Zweigniederlassung Schweiz, Opfikon
[11/28] Retrieved: UBS Foundation of Economics in Society
[12/28] Retrieved: UBS Fund Management (Switzerland) AG
[13/28] Retrieved: UBS Group AG
[14/28] Retrieved: UBS Hypotheken AG
[15/28] Retrieved: UBS Hypotheken Schweiz AG
[16/28] Retrieved: UBS Investment Foundation 1
[17/28] Retrieved: UBS Investment Foundation 2
[18/28] Retrieved: UBS Investment Foundation 3
[19/28] Retrieved: UBS Investment Foundation 4
[20/28] Retrieved: UBS Investment Foundation 5
[21/28] Retrieved: UBS Kulturstiftung
[22/28]

In [15]:
import pandas as pd

processed_data = []

for company in ubs_detailed_info:
    company_data = {
        'name': company.get('name', ''),
        'ehraid': company.get('ehraid', ''),
        'uid': company.get('uid', ''),
        'legalSeat': company.get('legalSeat', ''),
        'status': company.get('status', ''),
        'cantonalExcerptWeb': company.get('cantonalExcerptWeb', ''),
        'deleteDate': company.get('deleteDate', ''),
        'purpose': company.get('purpose', '')
    }
    # Get the English legal form name using our mapping
    legal_form_id = company.get('legalFormId', '')
    company_data['legalForm'] = legal_form_map.get(legal_form_id, '')
    # Use "or []" to avoid TypeError when value is None
    company_data['n_of_branchOffices'] = len(company.get('branchOffices') or [])
    company_data['n_of_hasTakenOver'] = len(company.get('hasTakenOver') or [])
    company_data['n_of_wasFoundedBy'] = len(company.get('wasFoundedBy') or [])
    company_data['n_of_wasMergedFrom'] = len(company.get('wasMergedFrom') or [])
    company_data['n_of_hasMoved'] = len(company.get('hasMoved') or [])
    processed_data.append(company_data)

ubs_df = pd.DataFrame(processed_data)
print("UBS Companies DataFrame:")
display(ubs_df)

UBS Companies DataFrame:


,name,ehraid,uid,legalSeat,status,cantonalExcerptWeb,deleteDate,purpose,legalForm,n_of_branchOffices,n_of_hasTakenOver,n_of_wasFoundedBy,n_of_wasMergedFrom,n_of_hasMoved
0,UBS 1e Sammelstiftung,1364537,CHE469254883,Schwyz,EXISTIEREND,https://sz.chregister.ch/cr-portal/auszug/ausz...,None,Der Zweck der Stiftung besteht in der überobli...,Foundation,0,0,0,0,0
1,UBS AG,415520,CHE101329561,Basel,EXISTIEREND,https://bs.chregister.ch/cr-portal/auszug/ausz...,None,Zweck der Gesellschaft ist der Betrieb einer B...,Corporation,9,14,0,0,0
2,UBS AG,421132,CHE101329561,Zürich,EXISTIEREND,https://zh.chregister.ch/cr-portal/auszug/ausz...,None,Zweck der Gesellschaft ist der Betrieb einer B...,Corporation,9,21,0,0,0
3,UBS Asset Management AG,1193524,CHE300147627,Zürich,EXISTIEREND,https://zh.chregister.ch/cr-portal/auszug/ausz...,None,"Zweck der Gesellschaft ist der Erwerb, das Hal...",Corporation,0,0,0,0,0
4,UBS Asset Management Switzerland AG,1372442,CHE153626718,Zürich,EXISTIEREND,https://zh.chregister.ch/cr-portal/auszug/ausz...,None,Zweck der Gesellschaft ist als Vermögensverwal...,Corporation,0,1,0,0,0
5,UBS Business Solutions AG,1236771,CHE262289477,Zürich,EXISTIEREND,https://zh.chregister.ch/cr-portal/auszug/ausz...,None,Zweck der Gesellschaft ist die Erbringung von ...,Corporation,0,1,0,0,0
6,UBS Card Center AG,450214,CHE102107595,Opfikon,EXISTIEREND,https://zh.chregister.ch/cr-portal/auszug/ausz...,None,Zweck der Gesellschaft ist die Erbringung von ...,Corporation,15,0,0,0,0
7,UBS Clean Energy Infrastructure Switzerland 2 AG,1319041,CHE313752134,Basel,EXISTIEREND,https://bs.chregister.ch/cr-portal/auszug/ausz...,None,Die Gesellschaft ist als Komplementärin für di...,Corporation,0,0,0,0,0
8,UBS Clean Energy Infrastructure Switzerland 2 ...,1321126,CHE249712999,Basel,EXISTIEREND,https://bs.chregister.ch/cr-portal/auszug/ausz...,None,Der ausschliessliche Zweck der Gesellschaft is...,Limited Partnership for collective investment ...,0,0,0,0,0
9,"UBS Europe SE, Frankfurt am Main, Zweigniederl...",596115,CHE101798341,Opfikon,EXISTIEREND,https://zh.chregister.ch/cr-portal/auszug/ausz...,None,Ausübung des Bankgeschäftes und aller anderen ...,Foreign branch,0,0,0,0,0


# Task2

### Step1. Retrieve Credit Suisse Details
Fetch the details of Credit Suisse using EHRAID 388851 and extract its `hasTakenOver` list.

### Step2. Recursive Search for All Taken-Over Companies
For each company in the `hasTakenOver` list, recursively retrieve its details and its own `hasTakenOver` list, tracking hops from Credit Suisse.

### Step3. Avoid Duplicates and Infinite Loops
Maintain a set of visited EHRAIDs to prevent reprocessing and infinite loops.

### Step4. Collect Required Information
For each company, collect the relevant fields along with the number of acquisition hops (`cs_hops`).

In [17]:
# Step 1: Retrieve Credit Suisse details and its direct takeovers

cs_ehraid = 388851
cs_url = f"https://www.zefix.ch/ZefixREST/api/v1/firm/{cs_ehraid}"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

response = requests.get(cs_url, headers=headers)
if response.status_code == 200:
    cs_info = response.json()
    print(f"Credit Suisse name: {cs_info.get('name', '')}")
    has_taken_over = cs_info.get('hasTakenOver') or []
    print(f"Number of companies directly taken over by Credit Suisse: {len(has_taken_over)}")
    # Optionally, print the first few for inspection
    for i, company in enumerate(has_taken_over[:3]):
        print(f"{i+1}. EHRAID: {company.get('ehraid', '')}, Name: {company.get('name', '')}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

Credit Suisse name: Credit Suisse AG
Number of companies directly taken over by Credit Suisse: 21
1. EHRAID: 619, Name: ABZ - Finanz- und Beteiligungsgesellschaft AG
2. EHRAID: 55686, Name: Faminta AG
3. EHRAID: 81396, Name: Hochhaus zur Palme AG


In [18]:
# Step 2: Recursively find all companies taken over by Credit Suisse

import time

def get_firm_details(ehraid):
    """Helper function to get firm details by EHRAID."""
    url = f"https://www.zefix.ch/ZefixREST/api/v1/firm/{ehraid}"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to get details for EHRAID {ehraid}")
        return None

def recursive_takeovers(starting_firms, cs_hops=1, visited=None, results=None):
    """
    Recursively find all companies taken over, directly or indirectly.
    - starting_firms: list of dicts (each with at least 'ehraid' and 'name')
    - cs_hops: current hop count from Credit Suisse
    - visited: set of already visited ehraids
    - results: list to collect results
    """
    if visited is None:
        visited = set()
    if results is None:
        results = []

    for firm in starting_firms:
        ehraid = firm.get('ehraid')
        if not ehraid or ehraid in visited:
            continue
        visited.add(ehraid)
        # Get full details
        details = get_firm_details(ehraid)
        if not details:
            continue
        # Collect required info
        result = {
            "name": details.get("name", ""),
            "ehraid": details.get("ehraid", ""),
            "legalSeat": details.get("legalSeat", ""),
            "legalFormId": details.get("legalFormId", ""),
            "status": details.get("status", ""),
            "cantonalExcerptWeb": details.get("cantonalExcerptWeb", ""),
            "deleteDate": details.get("deleteDate", ""),
            "cs_hops": cs_hops
        }
        results.append(result)
        # Recursively process its takeovers
        next_takeovers = details.get("hasTakenOver") or []
        if next_takeovers:
            recursive_takeovers(next_takeovers, cs_hops=cs_hops+1, visited=visited, results=results)
        # Optionally, sleep to avoid hammering the API
        time.sleep(0.1)
    return results

# Start recursion from Credit Suisse's direct takeovers
all_taken_over = recursive_takeovers(has_taken_over, cs_hops=1)

print(f"Total companies found (directly or indirectly taken over): {len(all_taken_over)}")
# Optionally, show a sample
for i, company in enumerate(all_taken_over[:3]):
    print(f"{i+1}. {company['name']} (EHRAID: {company['ehraid']}), cs_hops: {company['cs_hops']}")

Total companies found (directly or indirectly taken over): 40
1. ABZ - Finanz- und Beteiligungsgesellschaft AG (EHRAID: 619), cs_hops: 1
2. Faminta AG (EHRAID: 55686), cs_hops: 1
3. Hochhaus zur Palme AG (EHRAID: 81396), cs_hops: 1


In [19]:
# Step 3: Export the results as a JSON file

import json

output_filename = "credit_suisse_takeovers.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(all_taken_over, f, ensure_ascii=False, indent=2)

print(f"Exported {len(all_taken_over)} companies to {output_filename}")

Exported 40 companies to credit_suisse_takeovers.json


In [20]:
# Step 4: Convert the results to a pandas DataFrame

import pandas as pd

cs_takeovers_df = pd.DataFrame(all_taken_over)
print(f"DataFrame shape: {cs_takeovers_df.shape}")
display(cs_takeovers_df.head())

DataFrame shape: (40, 8)


,name,ehraid,legalSeat,legalFormId,status,cantonalExcerptWeb,deleteDate,cs_hops
0,ABZ - Finanz- und Beteiligungsgesellschaft AG,619,Zug,3,GELOESCHT,https://zg.chregister.ch/cr-portal/auszug/zefi...,2009-07-06,1
1,Faminta AG,55686,Zürich,3,GELOESCHT,https://zh.chregister.ch/cr-portal/auszug/zefi...,2008-04-04,1
2,Hochhaus zur Palme AG,81396,Zürich,3,GELOESCHT,https://zh.chregister.ch/cr-portal/auszug/zefi...,2006-07-06,1
3,Immobilien-Gesellschaft Glarus,86933,Glarus,3,GELOESCHT,https://gl.chregister.ch/cr-portal/auszug/zefi...,2006-07-06,1
4,Veryfinance AG,201418,Zug,3,GELOESCHT,https://zg.chregister.ch/cr-portal/auszug/zefi...,2004-08-17,1
